In [7]:
import numpy as np

# Variable size of submatrix (layer = size)
size = 3

# Input matrix
matrix = np.array([
    [1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1],
    [0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0],
    [0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1],
    [0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0],
    [0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1],
    [1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0]
])

rows, cols = matrix.shape
num_layers = rows // size

# Replace bypass (all-zero) submatrices with identity matrix
for i in range(0, rows, size):
    for j in range(0, cols, size):
        sub = matrix[i:i+size, j:j+size]
        if np.all(sub == 0):
            matrix[i:i+size, j:j+size] = np.eye(sub.shape[0], sub.shape[1])

def prev_one_in_row(row_ones, col):
    """Return the column index of the previous 1 in the row before 'col'.
    If none exists (col is leftmost 1), wrap around to the last 1 in the row."""
    before = [x for x in row_ones if x < col]
    if before:
        return max(before)   # nearest 1 to the left
    else:
        return max(row_ones) # wrap: last 1 in the row

# Generate edgelist:
# For each column, for each layer, find the column index of the previous 1
# in the same row as the current column's 1. Store as packed hex nibbles.
# Nibble[0] (MSN) = layer 0, Nibble[1] = layer 1, Nibble[2] = layer 2, ...
print("EdgeList (previous 1 per layer, packed as hex):")
for col in range(cols):
    nibbles = []
    for layer in range(num_layers):
        row_start = layer * size
        # Find which row in this layer has a 1 for this column
        rel_rows = np.where(matrix[row_start:row_start+size, col] == 1)[0]
        if len(rel_rows) == 0:
            nibbles.append(0)
            continue
        abs_row = row_start + rel_rows[0]
        # Get all columns in this row that have a 1
        row_ones = np.where(matrix[abs_row, :] == 1)[0].tolist()
        # Find the previous 1 in the row (with circular wrap)
        prev_idx = prev_one_in_row(row_ones, col)
        nibbles.append(prev_idx)
    # Pack nibbles into a single integer (MSN = layer 0)
    val = 0
    for nib in nibbles:
        val = (val << 4) | nib
    bits = num_layers * 4
    print(f"        edgelist[{col:2d}] = {bits}'h{val:0{num_layers}x};")

EdgeList (previous 1 per layer, packed as hex):
        edgelist[ 0] = 12'hb99;
        edgelist[ 1] = 12'h9aa;
        edgelist[ 2] = 12'habb;
        edgelist[ 3] = 12'h011;
        edgelist[ 4] = 12'h122;
        edgelist[ 5] = 12'h200;
        edgelist[ 6] = 12'h354;
        edgelist[ 7] = 12'h435;
        edgelist[ 8] = 12'h543;
        edgelist[ 9] = 12'h767;
        edgelist[10] = 12'h878;
        edgelist[11] = 12'h686;
